# Approach V5: 2. Feature Engineering (Expansion Phase)
**Project:** Honeywell Predictive Alerting Project - Tag: `03TIC_1023.PV` (Threshold: 21.0 °C)

This notebook generates a high-dimensional pool of candidate features directly within the notebook to show the engineering steps cell-by-cell.

In [1]:
import os
import pandas as pd
import numpy as np

## 1. Defining Preprocessing & Expansion Functions

In [2]:
DEFAULT_DATA_PATH = r"d:\Python-2025\Antigravity\honeywell\03TIC_1023_PVHI\03TIC_1023_PVHI\03TIC_1023_Final_merged_TripDataRemoved.parquet"

def load_and_preprocess_data(file_path=DEFAULT_DATA_PATH, target_col="03TIC_1023.PV"):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"DCS Historian dataset not found at: {file_path}")
        
    print("Loading raw historian data...")
    df = pd.read_parquet(file_path)
    
    if 'TimeStamp' in df.columns:
        df['TimeStamp'] = pd.to_datetime(df['TimeStamp'])
        df = df.sort_values('TimeStamp').set_index('TimeStamp')
    elif not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
        df = df.sort_index()
        
    print("Aligning index to uniform 1-minute grid...")
    full_idx = pd.date_range(start=df.index.min(), end=df.index.max(), freq='1min')
    df = df.reindex(full_idx)
    df.index.name = 'TimeStamp'
    
    print("Applying forward-fill imputation on short gaps (<= 5 mins)...")
    df = df.ffill(limit=5)
    return df

def engineer_features_pool(df, target_col="03TIC_1023.PV"):
    print("Starting Feature Engineering (Expansion)...")
    df_engineered = df.copy()
    
    key_cols = ['03TIC_1023.PV', '03TI_1024.PV', '03PIC_1023.PV', '03TI_1081.PV', '03TIC_1009.PV']
    
    # 1. Cyclical time context
    df_engineered['hour'] = df_engineered.index.hour
    df_engineered['month'] = df_engineered.index.month
    df_engineered['dayofweek'] = df_engineered.index.dayofweek
    
    # 2. Lag features
    print("  Generating lag features...")
    for col in key_cols:
        for lag in [1, 2, 5, 10, 15, 30, 60]:
            df_engineered[f'{col}_lag_{lag}'] = df_engineered[col].shift(lag)
            
    # 3. Rolling window statistics
    print("  Generating rolling window statistics...")
    for col in key_cols:
        for window in [10, 30, 60]:
            df_engineered[f'{col}_roll_mean_{window}'] = df_engineered[col].rolling(window=window, min_periods=1).mean()
            df_engineered[f'{col}_roll_std_{window}'] = df_engineered[col].rolling(window=window, min_periods=1).std()
            df_engineered[f'{col}_roll_max_{window}'] = df_engineered[col].rolling(window=window, min_periods=1).max()
            df_engineered[f'{col}_roll_min_{window}'] = df_engineered[col].rolling(window=window, min_periods=1).min()
            
    # 4. Expanding window statistics
    print("  Generating expanding statistics...")
    for col in key_cols:
        df_engineered[f'{col}_expanding_mean'] = df_engineered[col].expanding(min_periods=1).mean()
        df_engineered[f'{col}_expanding_max'] = df_engineered[col].expanding(min_periods=1).max()
        
    # 5. Rate-of-change / delta features
    print("  Generating delta rate-of-change features...")
    for col in key_cols:
        for diff in [5, 15, 30]:
            df_engineered[f'{col}_diff_{diff}'] = df_engineered[col] - df_engineered[col].shift(diff)
            
    # 6. Interaction terms
    print("  Generating thermodynamic interaction features...")
    df_engineered['temp_pressure_product'] = df_engineered['03TIC_1023.PV'] * df_engineered['03PIC_1023.PV']
    df_engineered['temp_delta_bottom_top'] = df_engineered['03TI_1024.PV'] - df_engineered['03TIC_1023.PV']
    
    df_engineered = df_engineered.dropna()
    print(f"Feature pool completed. Shape: {df_engineered.shape}")
    return df_engineered

## 2. Load Base Data

In [3]:
df = load_and_preprocess_data()
print(f"Data loaded. Shape: {df.shape}")

Loading raw historian data...


Aligning index to uniform 1-minute grid...


Applying forward-fill imputation on short gaps (<= 5 mins)...


Data loaded. Shape: (2112525, 19)


## 3. Generate Feature Pool

In [4]:
df_features = engineer_features_pool(df)

Starting Feature Engineering (Expansion)...


  Generating lag features...


  Generating rolling window statistics...


  Generating expanding statistics...


C:\Users\Ajay_ML\AppData\Local\Temp\ipykernel_19944\2257449723.py:56: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_engineered[f'{col}_expanding_max'] = df_engineered[col].expanding(min_periods=1).max()
C:\Users\Ajay_ML\AppData\Local\Temp\ipykernel_19944\2257449723.py:55: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_engineered[f'{col}_expanding_mean'] = df_engineered[col].expanding(min_periods=1).mean()


...

  Generating delta rate-of-change features...


C:\Users\Ajay_ML\AppData\Local\Temp\ipykernel_19944\2257449723.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_engineered[f'{col}_diff_{diff}'] = df_engineered[col] - df_engineered[col].shift(diff)
C:\Users\Ajay_ML\AppData\Local\Temp\ipykernel_19944\2257449723.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_engineered[f'{col}_diff_{diff}'] = df_engineered[col] - df_engineered[col].shift(diff)
C:\Users\Ajay_ML\AppData\Local\Temp\ipykernel_19944\2257449723.py:62: PerformanceWarning: DataFrame is highly fragmented. 

C:\Users\Ajay_ML\AppData\Local\Temp\ipykernel_19944\2257449723.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_engineered[f'{col}_diff_{diff}'] = df_engineered[col] - df_engineered[col].shift(diff)
C:\Users\Ajay_ML\AppData\Local\Temp\ipykernel_19944\2257449723.py:62: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_engineered[f'{col}_diff_{diff}'] = df_engineered[col] - df_engineered[col].shift(diff)
C:\Users\Ajay_ML\AppData\Local\Temp\ipykernel_19944\2257449723.py:66: PerformanceWarning: DataFrame is highly fragmented. 

  Generating thermodynamic interaction features...


Feature pool completed. Shape: (2013380, 144)


## 4. Save Candidate Features Pool

In [5]:
os.makedirs("outputs/v5", exist_ok=True)
df_features.to_parquet("outputs/v5/candidate_features_pool.parquet")
print("Saved candidate features pool to outputs/v5/candidate_features_pool.parquet")

Saved candidate features pool to outputs/v5/candidate_features_pool.parquet
